Homework 5

## Questions

(a) Explain self-attention. What are queries, keys, and values? Why is attention a
weighted sum?

(b) What is causal masking and why is it necessary in language models?

(c) Why are multiple attention heads used instead of a single head?

(d) What is the role of residual connections in transformer models?

### Explain self-attention. What are queries, keys, and values?

Self-attention is a mechanism, where every tokens use information from every other tokens 
with distribution of attention per every token.

Q, K, V is a matrix, that representate attention mechanism.

Q (query) is like search query, like "what i want to search?"

K (keys) is existing variants of results (keys)

V (value) contains information, if that value is picked.

example in other scope:

I want to search something in database. I make query (Q) and compared it with keys (K) of all rows.
Based on similarity i get weighted sum of values (V)

Why is attention a weighted sum?
Meaning of word ordinary depends on several words with different "weight". Word "school" depends not only on "student", 
but only on "teacher".
Also we can get a distribution of attention, due to weighted sum = 1.

### What is causal masking and why is it necessary in language models?

Causal masking is upper triangular matrix, that hide next (future) tokens for model (by multiply). 
It is necessary due to if we give model access to whole text, it we be trivial task to predict next token.
But this actual only for decoder-only models.

### Why are multiple attention heads used instead of a single head?

One head learn one representation of sentence. But there are a lot of cases, where exist two or more meaning.
Address this issue, we need to train several heads to catch all possible meaning per sentence.

### What is the role of residual connections in transformer models?

Residual connections helps to learn deep networks. If we have a lot of layers, 
gradients per early layer can be very small due to backpropogation (bad learning). Or for finite layer
gradients can be very large. (also bad learning)

To fix this, we can not only pass every layer in series, but also create a direct communication between far layers

## Implement Self-Attention

First we will import all libs:

In [6]:
from torch import nn
import torch
from torch.nn.functional import softmax
import math

And now we can create self-attention network:

In [28]:
class AttentitionNetwork(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)

    def forward(self, X):
        Q = self.Wq(X)
        K = self.Wk(X)
        V = self.Wv(X)

        res = (Q @ K.T) / math.sqrt(self.d)

        size = X.shape[0]
        mask = torch.triu(
            torch.ones(size, size),
            diagonal=1
        ).bool()

        res = res.masked_fill(mask, float('-inf'))

        A = softmax(res)
        output = A @ V

        return output, A

In [29]:
attention_model = AttentitionNetwork(5)

X = torch.randn(3, 5)

print('X in start:', X)

X in start: tensor([[ 0.1118, -0.5861, -0.1929, -0.1871,  0.2649],
        [ 0.3085,  1.8879,  0.7245, -0.1244,  2.3881],
        [ 0.4479,  0.0599, -0.1313,  0.3137, -1.4741]])


In [ ]:
output, A = attention_model(X)
print('output: ', output)
print('A: ', A)

output:  tensor([[ 0.2397, -0.0612,  0.2692, -0.0749,  0.1033],
        [-0.1866, -0.7865,  0.1986,  0.0824, -0.1842],
        [-0.0307, -0.0866, -0.1630,  0.0147,  0.0331]], grad_fn=<MmBackward0>)
A:  tensor([[1.0000, 0.0000, 0.0000],
        [0.3358, 0.6642, 0.0000],
        [0.3665, 0.2534, 0.3801]], grad_fn=<SoftmaxBackward0>)


/var/folders/y5/f3lzxvn542d8prr4rnvmc1dw0000gn/T/ipykernel_12388/1270969950.py:24: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  A = softmax(res)


We can see, matrix A is upper triangular matrix as causal masking require.